In [82]:
import igl
import numpy as np
import pyvista as pv
import ufl
from dolfinx import default_scalar_type, fem, io, plot
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem
from mpi4py import MPI

In [83]:
inner_radius_array = np.linspace(0.5, 1, 20)
outer_radius = 5
penalty_exp = 1
penalty_prefactor = 1

petsc_options = {
    "snes_type": "newtonls",
    "snes_max_it": 100,
    "snes_linesearch_type": "basic",
    "snes_atol": 1e-6,
    "snes_rtol": 1e-6,
    "snes_monitor": None,
    "ksp_max_it": 100,
    "ksp_error_if_not_converged": True,
    "ksp_type": "gmres",
    "ksp_rtol": 1e-8,
    "ksp_monitor": None,
    "pc_type": "hypre",
    "pc_hypre_type": "boomeramg",
    "pc_hypre_boomeramg_max_iter": 1,
    "pc_hypre_boomeramg_cycle_type": "v",
}

In [84]:
def map_path_to_circle(center, radius, orientation, start_angle, path):
    x_coordinates = np.zeros(len(path))
    y_coordinates = np.zeros(len(path))

    for i, _ in enumerate(path):
        angle = i / len(path) * 2 * np.pi * orientation + start_angle
        x_coordinates[i] = center[0] + radius * np.cos(angle)
        y_coordinates[i] = center[1] + radius * np.sin(angle)

    return x_coordinates, y_coordinates

In [85]:
def set_new_vertex_positions(mesh, vertices):
    mesh.geometry.x[:, :2] = vertices
    return mesh

def set_dirichlet_bcs(mesh, inner_radius):
    vertices = mesh.geometry.x[:, :2]
    simplices = mesh.topology.connectivity(mesh.topology.dim, 0).array
    boundary_loops = igl.boundary_loop_all(simplices.reshape(-1, 3))
    inner_boundary = np.array(boundary_loops[1])
    outer_boundary = np.array(boundary_loops[0])
    if vertices[inner_boundary[0], 0] < 1e-6:
        inner_boundary_start_angle = np.pi/2
    else:
        inner_boundary_start_angle = np.arctan(
            vertices[inner_boundary[0], 1] / vertices[inner_boundary[0], 0]
        )
        inner_boundary_start_angle = (inner_boundary_start_angle * 180 / np.pi + 180) * np.pi / 180
    if vertices[outer_boundary[0], 0] < 1e-6:
        outer_boundary_start_angle = np.pi/2
    else:
        outer_boundary_start_angle = np.arctan(
            vertices[outer_boundary[0], 1] / vertices[outer_boundary[0], 0]
        )
        outer_boundary_start_angle = (outer_boundary_start_angle * 180 / np.pi + 180) * np.pi / 180
    inner_bc_values = np.vstack(
        map_path_to_circle((0, 0), inner_radius, -1, inner_boundary_start_angle, inner_boundary)
    ).T
    outer_bc_values = np.vstack(
        map_path_to_circle((0, 0), outer_radius, 1, outer_boundary_start_angle, outer_boundary)
    ).T
    inner_boundary_function = fem.Function(funcspace)
    inner_boundary_array = inner_boundary_function.x.array.reshape(-1, 2)
    inner_boundary_array[inner_boundary] = inner_bc_values
    inner_boundary_function.x.array[:] = inner_boundary_array.flatten()
    outer_boundary_function = fem.Function(funcspace)
    outer_boundary_array = outer_boundary_function.x.array.reshape(-1, 2)
    outer_boundary_array[outer_boundary] = outer_bc_values
    outer_boundary_function.x.array[:] = outer_boundary_array.flatten()
    inner_bc = fem.dirichletbc(inner_boundary_function, inner_boundary)
    outer_bc = fem.dirichletbc(outer_boundary_function, outer_boundary)
    print(vertices[inner_boundary[0], 0], vertices[outer_boundary[0], 0])
    return inner_bc, outer_bc

In [86]:
def compute_penalty_term(mesh):
    DG0 = fem.functionspace(mesh, ("DG", 0))
    v_scalar = ufl.TestFunction(DG0)
    cell_area_form = fem.form(ufl.inner(fem.Constant(mesh, 1.0), v_scalar) * ufl.dx(domain=mesh))
    cell_areas = fem.assemble_vector(cell_area_form)
    cell_area_function = fem.Function(DG0)
    cell_area_function.x.array[:] = np.power(
        np.average(cell_areas.array) / cell_areas.array, penalty_exp
    )
    f_scalar = fem.functionspace(mesh, ("Lagrange", 1, ()))
    u_scalar = ufl.TrialFunction(f_scalar)
    v_scalar = ufl.TestFunction(f_scalar)
    a = ufl.inner(u_scalar, v_scalar) * ufl.dx
    L = ufl.inner(cell_area_function, v_scalar) * ufl.dx
    problem = LinearProblem(a, L, bcs=[], petsc_options_prefix="projection")
    penalty_term = problem.solve()
    return penalty_term

def assemble_residual_form(funcspace, penalty_term):
    v = ufl.TestFunction(funcspace)
    u = fem.Function(funcspace)
    d = len(u)

    I = ufl.variable(ufl.Identity(d))
    F = ufl.variable(I + ufl.grad(u))
    C = ufl.variable(F.T * F)
    Ic = ufl.variable(ufl.tr(C))
    J = ufl.variable(ufl.det(F))

    E = 1e4
    nu = default_scalar_type(0.3)
    mu = E / (2 * (1 + nu))
    lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
    psi = (
        (mu / 2) * (Ic - 3)
        - mu * ufl.ln(J)
        + (lmbda / 2) * (ufl.ln(J)) ** 2
        - penalty_prefactor * penalty_term * ufl.ln(J)
    )
    P = ufl.diff(psi, F)

    dx = ufl.Measure("dx", domain=mesh)
    residual = ufl.inner(ufl.grad(v), P) * dx
    return u, residual

def solve_nonlinear_problem(residual_form, u, inner_bc, outer_bc):
    problem = NonlinearProblem(
        residual_form,
        u,
        bcs=[inner_bc, outer_bc],
        petsc_options=petsc_options,
        petsc_options_prefix="elasticity",
    )
    problem.solve()
    return u.x.array.reshape(-1, 2)

In [87]:
np.min(penalty_term.x.array)

np.float64(0.013601073084544042)

In [88]:
with io.XDMFFile(MPI.COMM_WORLD, "uac_mesh_1.xdmf", "r") as xdmf:
    mesh = xdmf.read_mesh(name="Grid")
mesh.topology.create_connectivity(mesh.topology.dim, mesh.topology.dim - 1)
mesh.topology.create_connectivity(mesh.topology.dim-1, mesh.topology.dim)
funcspace = fem.functionspace(mesh, ("CG", 1, (2,)))

for inner_radius in inner_radius_array:
    print(inner_radius)
    inner_bc, outer_bc = set_dirichlet_bcs(mesh, inner_radius)
    penalty_term = compute_penalty_term(mesh)
    u, residual_form = assemble_residual_form(funcspace, penalty_term)
    displaced_vertices = solve_nonlinear_problem(residual_form, u, inner_bc, outer_bc)
    mesh = set_new_vertex_positions(mesh, displaced_vertices)

0.5
-0.09666372685983783 -1.5924332512584216
  0 SNES Function norm 6.128859641832e+05
    Residual norms for elasticity solve.
    0 KSP Residual norm 1.071272782514e+02
    1 KSP Residual norm 4.876418786292e+00
    2 KSP Residual norm 2.721475988097e-01
    3 KSP Residual norm 2.327713156590e-02
    4 KSP Residual norm 1.282540669709e-03
    5 KSP Residual norm 1.058558586193e-04
    6 KSP Residual norm 8.671368048928e-06
    7 KSP Residual norm 5.626268866540e-07
  1 SNES Function norm 2.693965972179e+02
    Residual norms for elasticity solve.
    0 KSP Residual norm 7.375413833969e-02
    1 KSP Residual norm 3.297791192009e-03
    2 KSP Residual norm 1.480251845269e-04
    3 KSP Residual norm 6.737903777858e-06
    4 KSP Residual norm 2.577213661642e-07
    5 KSP Residual norm 1.191855883059e-08
    6 KSP Residual norm 4.930540750855e-10
  2 SNES Function norm 2.601098864056e+00
    Residual norms for elasticity solve.
    0 KSP Residual norm 9.442956614789e-04
    1 KSP Residual

In [89]:
simplices, types, vertices = plot.vtk_mesh(mesh,dim=2)
pv_mesh = pv.UnstructuredGrid(simplices, types, np.hstack([displaced_vertices, np.zeros((len(vertices), 1))]))
pv.save_meshio("displaced_mesh.vtk", pv_mesh)

In [90]:
displaced_vertices

array([[ 3.06161700e-16,  5.00000000e+00],
       [ 2.30939368e-01,  4.61788977e+00],
       [ 4.97839233e-01,  4.97515388e+00],
       ...,
       [ 9.08061094e-04, -1.44858885e+00],
       [-1.27803394e-01, -1.49338376e+00],
       [-1.36136547e-01, -1.40498392e+00]], shape=(1946, 2))